# 08 - Silver: Country and Bloc Dimensions

This notebook creates the core silver dimensions used to reconcile country identifiers across all bronze sources.

Outputs:

- `silver.dim_country`
- `silver.dim_bloc_membership`

The country dimension is intentionally static and explicit. Bronze sources use different identifier systems: ISO3, ISO2, UN M49, World Bank IDs, IMF ISO3, ACLED numeric ISO, and source-specific country names. This table is the canonical bridge.

The bloc membership table uses a primary analytical classification for dashboard grouping. For Burkina Faso, Mali, and Niger, the AES analytical period starts on 2024-07-06, when the AES confederation treaty was adopted. ECOWAS legal withdrawal became effective on 2025-01-29, so this table preserves that distinction in the notes column.

In [ ]:
# Cell 1 - Configuration and canonical country rows
from datetime import datetime, timezone

from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("USE SCHEMA silver")

loaded_at = datetime.now(timezone.utc)

PROJECT_COUNTRIES = [
    {
        "country_key": 1,
        "country_iso3": "CMR",
        "country_iso2": "CM",
        "m49_code": 120,
        "country_name": "Cameroon",
        "official_name": "Republic of Cameroon",
        "region": "Central Africa",
        "current_primary_bloc_code": "CEMAC",
        "current_primary_bloc_name": "Economic and Monetary Community of Central Africa",
        "is_cemac_current": True,
        "is_ecowas_current": False,
        "is_aes_current": False,
        "wb_country_id": "CM",
        "imf_iso3": "CMR",
        "source_name_variants": ["Cameroon"],
    },
    {
        "country_key": 2,
        "country_iso3": "CAF",
        "country_iso2": "CF",
        "m49_code": 140,
        "country_name": "Central African Republic",
        "official_name": "Central African Republic",
        "region": "Central Africa",
        "current_primary_bloc_code": "CEMAC",
        "current_primary_bloc_name": "Economic and Monetary Community of Central Africa",
        "is_cemac_current": True,
        "is_ecowas_current": False,
        "is_aes_current": False,
        "wb_country_id": "CF",
        "imf_iso3": "CAF",
        "source_name_variants": ["Central African Republic", "Central African Rep."],
    },
    {
        "country_key": 3,
        "country_iso3": "TCD",
        "country_iso2": "TD",
        "m49_code": 148,
        "country_name": "Chad",
        "official_name": "Republic of Chad",
        "region": "Central Africa",
        "current_primary_bloc_code": "CEMAC",
        "current_primary_bloc_name": "Economic and Monetary Community of Central Africa",
        "is_cemac_current": True,
        "is_ecowas_current": False,
        "is_aes_current": False,
        "wb_country_id": "TD",
        "imf_iso3": "TCD",
        "source_name_variants": ["Chad"],
    },
    {
        "country_key": 4,
        "country_iso3": "COG",
        "country_iso2": "CG",
        "m49_code": 178,
        "country_name": "Congo, Rep.",
        "official_name": "Republic of the Congo",
        "region": "Central Africa",
        "current_primary_bloc_code": "CEMAC",
        "current_primary_bloc_name": "Economic and Monetary Community of Central Africa",
        "is_cemac_current": True,
        "is_ecowas_current": False,
        "is_aes_current": False,
        "wb_country_id": "CG",
        "imf_iso3": "COG",
        "source_name_variants": ["Congo, Rep.", "Republic of Congo", "Republic of the Congo", "Congo Republic"],
    },
    {
        "country_key": 5,
        "country_iso3": "GNQ",
        "country_iso2": "GQ",
        "m49_code": 226,
        "country_name": "Equatorial Guinea",
        "official_name": "Republic of Equatorial Guinea",
        "region": "Central Africa",
        "current_primary_bloc_code": "CEMAC",
        "current_primary_bloc_name": "Economic and Monetary Community of Central Africa",
        "is_cemac_current": True,
        "is_ecowas_current": False,
        "is_aes_current": False,
        "wb_country_id": "GQ",
        "imf_iso3": "GNQ",
        "source_name_variants": ["Equatorial Guinea"],
    },
    {
        "country_key": 6,
        "country_iso3": "GAB",
        "country_iso2": "GA",
        "m49_code": 266,
        "country_name": "Gabon",
        "official_name": "Gabonese Republic",
        "region": "Central Africa",
        "current_primary_bloc_code": "CEMAC",
        "current_primary_bloc_name": "Economic and Monetary Community of Central Africa",
        "is_cemac_current": True,
        "is_ecowas_current": False,
        "is_aes_current": False,
        "wb_country_id": "GA",
        "imf_iso3": "GAB",
        "source_name_variants": ["Gabon"],
    },
    {
        "country_key": 7,
        "country_iso3": "BEN",
        "country_iso2": "BJ",
        "m49_code": 204,
        "country_name": "Benin",
        "official_name": "Republic of Benin",
        "region": "West Africa",
        "current_primary_bloc_code": "ECOWAS",
        "current_primary_bloc_name": "Economic Community of West African States",
        "is_cemac_current": False,
        "is_ecowas_current": True,
        "is_aes_current": False,
        "wb_country_id": "BJ",
        "imf_iso3": "BEN",
        "source_name_variants": ["Benin"],
    },
    {
        "country_key": 8,
        "country_iso3": "BFA",
        "country_iso2": "BF",
        "m49_code": 854,
        "country_name": "Burkina Faso",
        "official_name": "Burkina Faso",
        "region": "West Africa",
        "current_primary_bloc_code": "AES",
        "current_primary_bloc_name": "Alliance of Sahel States",
        "is_cemac_current": False,
        "is_ecowas_current": False,
        "is_aes_current": True,
        "wb_country_id": "BF",
        "imf_iso3": "BFA",
        "source_name_variants": ["Burkina Faso"],
    },
    {
        "country_key": 9,
        "country_iso3": "CPV",
        "country_iso2": "CV",
        "m49_code": 132,
        "country_name": "Cabo Verde",
        "official_name": "Republic of Cabo Verde",
        "region": "West Africa",
        "current_primary_bloc_code": "ECOWAS",
        "current_primary_bloc_name": "Economic Community of West African States",
        "is_cemac_current": False,
        "is_ecowas_current": True,
        "is_aes_current": False,
        "wb_country_id": "CV",
        "imf_iso3": "CPV",
        "source_name_variants": ["Cabo Verde", "Cape Verde"],
    },
    {
        "country_key": 10,
        "country_iso3": "CIV",
        "country_iso2": "CI",
        "m49_code": 384,
        "country_name": "Cote d'Ivoire",
        "official_name": "Republic of Cote d'Ivoire",
        "region": "West Africa",
        "current_primary_bloc_code": "ECOWAS",
        "current_primary_bloc_name": "Economic Community of West African States",
        "is_cemac_current": False,
        "is_ecowas_current": True,
        "is_aes_current": False,
        "wb_country_id": "CI",
        "imf_iso3": "CIV",
        "source_name_variants": ["Cote d'Ivoire", "Ivory Coast"],
    },
    {
        "country_key": 11,
        "country_iso3": "GMB",
        "country_iso2": "GM",
        "m49_code": 270,
        "country_name": "Gambia",
        "official_name": "Republic of The Gambia",
        "region": "West Africa",
        "current_primary_bloc_code": "ECOWAS",
        "current_primary_bloc_name": "Economic Community of West African States",
        "is_cemac_current": False,
        "is_ecowas_current": True,
        "is_aes_current": False,
        "wb_country_id": "GM",
        "imf_iso3": "GMB",
        "source_name_variants": ["Gambia", "Gambia, The", "The Gambia"],
    },
    {
        "country_key": 12,
        "country_iso3": "GHA",
        "country_iso2": "GH",
        "m49_code": 288,
        "country_name": "Ghana",
        "official_name": "Republic of Ghana",
        "region": "West Africa",
        "current_primary_bloc_code": "ECOWAS",
        "current_primary_bloc_name": "Economic Community of West African States",
        "is_cemac_current": False,
        "is_ecowas_current": True,
        "is_aes_current": False,
        "wb_country_id": "GH",
        "imf_iso3": "GHA",
        "source_name_variants": ["Ghana"],
    },
    {
        "country_key": 13,
        "country_iso3": "GIN",
        "country_iso2": "GN",
        "m49_code": 324,
        "country_name": "Guinea",
        "official_name": "Republic of Guinea",
        "region": "West Africa",
        "current_primary_bloc_code": "ECOWAS",
        "current_primary_bloc_name": "Economic Community of West African States",
        "is_cemac_current": False,
        "is_ecowas_current": True,
        "is_aes_current": False,
        "wb_country_id": "GN",
        "imf_iso3": "GIN",
        "source_name_variants": ["Guinea"],
    },
    {
        "country_key": 14,
        "country_iso3": "GNB",
        "country_iso2": "GW",
        "m49_code": 624,
        "country_name": "Guinea-Bissau",
        "official_name": "Republic of Guinea-Bissau",
        "region": "West Africa",
        "current_primary_bloc_code": "ECOWAS",
        "current_primary_bloc_name": "Economic Community of West African States",
        "is_cemac_current": False,
        "is_ecowas_current": True,
        "is_aes_current": False,
        "wb_country_id": "GW",
        "imf_iso3": "GNB",
        "source_name_variants": ["Guinea-Bissau", "Guinea Bissau"],
    },
    {
        "country_key": 15,
        "country_iso3": "LBR",
        "country_iso2": "LR",
        "m49_code": 430,
        "country_name": "Liberia",
        "official_name": "Republic of Liberia",
        "region": "West Africa",
        "current_primary_bloc_code": "ECOWAS",
        "current_primary_bloc_name": "Economic Community of West African States",
        "is_cemac_current": False,
        "is_ecowas_current": True,
        "is_aes_current": False,
        "wb_country_id": "LR",
        "imf_iso3": "LBR",
        "source_name_variants": ["Liberia"],
    },
    {
        "country_key": 16,
        "country_iso3": "MLI",
        "country_iso2": "ML",
        "m49_code": 466,
        "country_name": "Mali",
        "official_name": "Republic of Mali",
        "region": "West Africa",
        "current_primary_bloc_code": "AES",
        "current_primary_bloc_name": "Alliance of Sahel States",
        "is_cemac_current": False,
        "is_ecowas_current": False,
        "is_aes_current": True,
        "wb_country_id": "ML",
        "imf_iso3": "MLI",
        "source_name_variants": ["Mali"],
    },
    {
        "country_key": 17,
        "country_iso3": "NER",
        "country_iso2": "NE",
        "m49_code": 562,
        "country_name": "Niger",
        "official_name": "Republic of Niger",
        "region": "West Africa",
        "current_primary_bloc_code": "AES",
        "current_primary_bloc_name": "Alliance of Sahel States",
        "is_cemac_current": False,
        "is_ecowas_current": False,
        "is_aes_current": True,
        "wb_country_id": "NE",
        "imf_iso3": "NER",
        "source_name_variants": ["Niger"],
    },
    {
        "country_key": 18,
        "country_iso3": "NGA",
        "country_iso2": "NG",
        "m49_code": 566,
        "country_name": "Nigeria",
        "official_name": "Federal Republic of Nigeria",
        "region": "West Africa",
        "current_primary_bloc_code": "ECOWAS",
        "current_primary_bloc_name": "Economic Community of West African States",
        "is_cemac_current": False,
        "is_ecowas_current": True,
        "is_aes_current": False,
        "wb_country_id": "NG",
        "imf_iso3": "NGA",
        "source_name_variants": ["Nigeria"],
    },
    {
        "country_key": 19,
        "country_iso3": "SEN",
        "country_iso2": "SN",
        "m49_code": 686,
        "country_name": "Senegal",
        "official_name": "Republic of Senegal",
        "region": "West Africa",
        "current_primary_bloc_code": "ECOWAS",
        "current_primary_bloc_name": "Economic Community of West African States",
        "is_cemac_current": False,
        "is_ecowas_current": True,
        "is_aes_current": False,
        "wb_country_id": "SN",
        "imf_iso3": "SEN",
        "source_name_variants": ["Senegal"],
    },
    {
        "country_key": 20,
        "country_iso3": "SLE",
        "country_iso2": "SL",
        "m49_code": 694,
        "country_name": "Sierra Leone",
        "official_name": "Republic of Sierra Leone",
        "region": "West Africa",
        "current_primary_bloc_code": "ECOWAS",
        "current_primary_bloc_name": "Economic Community of West African States",
        "is_cemac_current": False,
        "is_ecowas_current": True,
        "is_aes_current": False,
        "wb_country_id": "SL",
        "imf_iso3": "SLE",
        "source_name_variants": ["Sierra Leone"],
    },
    {
        "country_key": 21,
        "country_iso3": "TGO",
        "country_iso2": "TG",
        "m49_code": 768,
        "country_name": "Togo",
        "official_name": "Togolese Republic",
        "region": "West Africa",
        "current_primary_bloc_code": "ECOWAS",
        "current_primary_bloc_name": "Economic Community of West African States",
        "is_cemac_current": False,
        "is_ecowas_current": True,
        "is_aes_current": False,
        "wb_country_id": "TG",
        "imf_iso3": "TGO",
        "source_name_variants": ["Togo"],
    },
]

country_df = spark.createDataFrame(PROJECT_COUNTRIES).withColumn("loaded_at", F.lit(loaded_at))

print(f"Canonical countries: {country_df.count()}")
country_df.orderBy("country_key").show(25, truncate=False)

In [ ]:
# Cell 2 - Validate and write silver.dim_country
expected_country_count = 21

checks = {
    "row_count": country_df.count(),
    "distinct_iso3": country_df.select("country_iso3").distinct().count(),
    "distinct_iso2": country_df.select("country_iso2").distinct().count(),
    "distinct_m49": country_df.select("m49_code").distinct().count(),
    "distinct_country_key": country_df.select("country_key").distinct().count(),
}

for check_name, value in checks.items():
    print(f"{check_name}: {value}")
    if value != expected_country_count:
        raise ValueError(f"{check_name} expected {expected_country_count}, got {value}")

spark.sql("DROP TABLE IF EXISTS silver.dim_country")

(country_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_country"))

print("Write complete: silver.dim_country")

In [ ]:
# Cell 3 - Build and write silver.dim_bloc_membership
CEMAC_ISO3 = {"CMR", "CAF", "TCD", "COG", "GNQ", "GAB"}
AES_ISO3 = {"BFA", "MLI", "NER"}

BLOCS = {
    "CEMAC": "Economic and Monetary Community of Central Africa",
    "ECOWAS": "Economic Community of West African States",
    "AES": "Alliance of Sahel States",
}

bloc_rows = []

for country in PROJECT_COUNTRIES:
    iso3 = country["country_iso3"]
    name = country["country_name"]

    if iso3 in CEMAC_ISO3:
        bloc_rows.append({
            "country_iso3": iso3,
            "country_name": name,
            "bloc_code": "CEMAC",
            "bloc_name": BLOCS["CEMAC"],
            "valid_from": "1990-01-01",
            "valid_to": None,
            "is_current": True,
            "is_primary_analytical_bloc": True,
            "membership_basis": "project_current_member_backcast",
            "source_note": "Current CEMAC member backcast to the analytical start year for longitudinal comparison.",
        })
    elif iso3 in AES_ISO3:
        bloc_rows.append({
            "country_iso3": iso3,
            "country_name": name,
            "bloc_code": "ECOWAS",
            "bloc_name": BLOCS["ECOWAS"],
            "valid_from": "1990-01-01",
            "valid_to": "2024-07-05",
            "is_current": False,
            "is_primary_analytical_bloc": True,
            "membership_basis": "project_primary_analytical_bloc",
            "source_note": "Analytical ECOWAS period before AES confederation. Legal ECOWAS withdrawal became effective on 2025-01-29.",
        })
        bloc_rows.append({
            "country_iso3": iso3,
            "country_name": name,
            "bloc_code": "AES",
            "bloc_name": BLOCS["AES"],
            "valid_from": "2024-07-06",
            "valid_to": None,
            "is_current": True,
            "is_primary_analytical_bloc": True,
            "membership_basis": "project_primary_analytical_bloc",
            "source_note": "AES confederation treaty adopted on 2024-07-06. Legal ECOWAS withdrawal became effective on 2025-01-29.",
        })
    else:
        bloc_rows.append({
            "country_iso3": iso3,
            "country_name": name,
            "bloc_code": "ECOWAS",
            "bloc_name": BLOCS["ECOWAS"],
            "valid_from": "1990-01-01",
            "valid_to": None,
            "is_current": True,
            "is_primary_analytical_bloc": True,
            "membership_basis": "project_current_member_backcast",
            "source_note": "Current ECOWAS member backcast to the analytical start year for longitudinal comparison.",
        })

bloc_df = (
    spark.createDataFrame(bloc_rows)
    .withColumn("valid_from", F.to_date("valid_from"))
    .withColumn("valid_to", F.to_date("valid_to"))
    .withColumn("loaded_at", F.lit(loaded_at))
)

spark.sql("DROP TABLE IF EXISTS silver.dim_bloc_membership")

(bloc_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_bloc_membership"))

print("Write complete: silver.dim_bloc_membership")
bloc_df.orderBy("bloc_code", "country_iso3", "valid_from").show(40, truncate=False)

In [ ]:
# Cell 4 - Dimension verification
print("dim_country coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS countries,
        COUNT(DISTINCT country_iso3) AS iso3_codes,
        COUNT(DISTINCT m49_code) AS m49_codes,
        SUM(CASE WHEN current_primary_bloc_code = 'CEMAC' THEN 1 ELSE 0 END) AS cemac_countries,
        SUM(CASE WHEN current_primary_bloc_code = 'ECOWAS' THEN 1 ELSE 0 END) AS ecowas_countries,
        SUM(CASE WHEN current_primary_bloc_code = 'AES' THEN 1 ELSE 0 END) AS aes_countries
    FROM silver.dim_country
""").show()

print("Current primary analytical blocs:")
spark.sql("""
    SELECT
        current_primary_bloc_code AS bloc_code,
        current_primary_bloc_name AS bloc_name,
        COUNT(*) AS countries,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(country_iso3))) AS iso3_list
    FROM silver.dim_country
    GROUP BY current_primary_bloc_code, current_primary_bloc_name
    ORDER BY bloc_code
""").show(truncate=False)

print("Bloc membership intervals:")
spark.sql("""
    SELECT
        bloc_code,
        country_iso3,
        country_name,
        valid_from,
        valid_to,
        is_current,
        membership_basis
    FROM silver.dim_bloc_membership
    ORDER BY country_iso3, valid_from
""").show(40, truncate=False)

In [ ]:
# Cell 5 - Check country coverage in bronze source tables
def table_exists(schema_name, table_name):
    return spark.sql(f"SHOW TABLES IN {schema_name} LIKE '{table_name}'").count() > 0


bronze_country_columns = [
    ("data360_raw", "country_iso3", "World Bank Data360"),
    ("imts_raw", "reporter_iso3", "IMF IMTS reporter countries"),
    ("acled_events_historical", "iso3", "ACLED events"),
    ("acled_weekly_aggregated", "iso3", "ACLED weekly aggregates"),
    ("imf_weo_raw", "country_iso3", "IMF WEO"),
    ("fsi_raw", "country_iso3", "Fragile States Index"),
]

dim_iso3_df = spark.table("silver.dim_country").select("country_iso3").distinct()
coverage_rows = []

for table_name, country_col, label in bronze_country_columns:
    full_name = f"bronze.{table_name}"

    if not table_exists("bronze", table_name):
        coverage_rows.append({
            "source_table": full_name,
            "source_label": label,
            "status": "MISSING_TABLE",
            "countries_in_source": 0,
            "missing_from_source": 21,
            "unknown_to_dimension": 0,
        })
        continue

    source_iso3_df = (
        spark.table(full_name)
        .select(F.col(country_col).cast("string").alias("country_iso3"))
        .where(F.col("country_iso3").isNotNull() & (F.col("country_iso3") != ""))
        .distinct()
    )

    missing_from_source = dim_iso3_df.join(source_iso3_df, "country_iso3", "left_anti").count()
    unknown_to_dimension = source_iso3_df.join(dim_iso3_df, "country_iso3", "left_anti").count()
    countries_in_source = source_iso3_df.count()

    status = "OK" if missing_from_source == 0 and unknown_to_dimension == 0 else "CHECK"

    coverage_rows.append({
        "source_table": full_name,
        "source_label": label,
        "status": status,
        "countries_in_source": countries_in_source,
        "missing_from_source": missing_from_source,
        "unknown_to_dimension": unknown_to_dimension,
    })

coverage_df = spark.createDataFrame(coverage_rows)
coverage_df.orderBy("source_table").show(truncate=False)

if coverage_df.where(F.col("status") != "OK").count() > 0:
    print("One or more bronze tables need inspection. Missing tables are expected only if that source has not been loaded yet.")

In [ ]:
# Cell 6 - Example join pattern for later silver fact notebooks
print("Latest FSI total score joined to current primary bloc:")
if table_exists("bronze", "fsi_raw"):
    spark.sql("""
        SELECT
            c.current_primary_bloc_code AS bloc_code,
            c.country_iso3,
            c.country_name,
            f.year,
            ROUND(f.value, 1) AS fsi_total_score
        FROM bronze.fsi_raw f
        JOIN silver.dim_country c
          ON f.country_iso3 = c.country_iso3
        WHERE f.indicator_code = 'TOTAL'
          AND f.year = (SELECT MAX(year) FROM bronze.fsi_raw)
        ORDER BY f.value DESC
    """).show(25, truncate=False)
else:
    print("bronze.fsi_raw not found yet; run 07_bronze_fsi_extract first.")